<a href="https://colab.research.google.com/github/kligeys/AI-Projects/blob/main/Multiagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import json
from openai import OpenAI
from google.colab import userdata
import yfinance as yf
import os
import requests

In [33]:
# 1. Initialize API Keys
# It's best practice to use environment variables, but you can hardcode them for testing.
# 1. Retrieve the API key safely from Colab Secrets
api_key = userdata.get('OPENAI_API_KEY')
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", api_key)
weather_api_key = userdata.get('OPENWEATHER_API_KEY')
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY", weather_api_key)

In [14]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [34]:
# 2. Define the Weather Tool Function
def get_weather_forecast(location):
    """Fetches real-time weather information using the OpenWeatherMap API."""
    base_url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": location,
        "appid": weather_api_key,
        "units": "metric",  # Use 'imperial' for Fahrenheit
    }

    try:
        response = requests.get(base_url, params=params)

        if response.status_code == 200:
            data = response.json()
            weather_desc = data["weather"][0]["description"]
            temp = data["main"]["temp"]
            humidity = data["main"]["humidity"]
            return json.dumps(
                {
                    "location": location,
                    "temperature": f"{temp}°C",
                    "condition": weather_desc,
                    "humidity": f"{humidity}%",
                },
                ensure_ascii=False # Add this to prevent Unicode escaping
            )
        else:
            error_message = f"Could not find weather for '{location}'. Status Code: {response.status_code}. Response: {response.text}"
            print(f"DEBUG: Weather API error: {error_message}") # Add a debug print
            return json.dumps(
                {"error": error_message},
                ensure_ascii=False # Add this to prevent Unicode escaping
            )
    except Exception as e:
        print(f"DEBUG: Weather API exception: {str(e)}") # Add a debug print
        return json.dumps({"error": str(e)}, ensure_ascii=False)
#

In [35]:
# Define the tool function
def get_stock_price(ticker):
  """Fetches the current stock price for a given ticker symbol using Yahoo Finance."""
  stock = yf.Ticker(ticker)
  # Get current price from fast_info or history
  price = stock.fast_info.get("lastPrice", None)
  if price is None:
    hist = stock.history(period="1d")
    if not hist.empty:
      price = hist["Close"].iloc[-1]
  if price:
    return json.dumps({"ticker": ticker.upper(), "price": round(price, 2)})
  return json.dumps({"error": f"Could not find price for {ticker}"})


In [22]:
# Map string names to the actual function objects for easy execution
TOOL_MAP = {"get_weather_forecast": get_weather_forecast, "get_stock_price": get_stock_price}

In [36]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather_forecast",
            "description": "Get the current weather or forecast for a given city/location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state/country, e.g., San Francisco, CA or Paris, France",
                    }
                },
                "required": ["location"],
            },
        },
    },
    {
    "type": "function",
    "function": {
        "name": "get_stock_price",
        "description": "Get the current stock price for a company ticker symbol",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "The stock ticker symbol, e.g. AAPL, TSLA",
                }
            },
            "required": ["ticker"],
        },
    },
}
]

In [28]:
#Step 3: Create the Multi-Agent Execution Loop
def run_multi_agent_system(user_prompt):
    print(f"User Prompt: '{user_prompt}'\n")

    # Initialize conversation history
    messages = [
        {
            "role": "system",
            "content": "You are a helpful coordinator agent. Delegate work to your tools to answer the user.",
        },
        {"role": "user", "content": user_prompt},
    ]

    # Step 1: Call the model with tools enabled
    response = client.chat.completions.create(
        model="gpt-4o",  # or "gpt-4-turbo"
        messages=messages,
        tools=tools_schema,
        tool_choice="auto",
    )

    response_message = response.choices[0].message
    print("the response is ", response_message)
    tool_calls = response_message.tool_calls

    # Step 2: Check if the model requested any tool executions
    if tool_calls:
        # Append the assistant's request to call tools to the history
        messages.append(response_message)

        # Process each tool call sequentially (No asyncio)
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            print(
                f"🤖 Coordinator routed task to -> {function_name}({function_args})"
            )

            # Get the actual Python function from our map and run it
            function_to_call = TOOL_MAP[function_name]
            tool_output = function_to_call(**function_args)

            # Append the tool result back to the message history
            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": tool_output,
                }
            )
            # Step 3: Get final response from the agent with the new context
        final_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
        )

        return final_response.choices[0].message.content
    else:
        # If no tools were required, return the initial response
        return response_message.content


In [37]:
#Step 4: Run the System
if __name__ == "__main__":
    prompt = (
        "What is the weather like in Tokyo right now, and what is the current price of AAPL stock?"
    )
    result = run_multi_agent_system(prompt)

    print("\n✨ Final Agent Response:")
    print(result)


User Prompt: 'What is the weather like in Tokyo right now, and what is the current price of AAPL stock?'

the response is  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_vJuEHpJTn6jaN2PykLlYHx7W', function=Function(arguments='{"location": "Tokyo, Japan"}', name='get_weather_forecast'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_rqUh98vgqXxKlgTQ5lcU12PY', function=Function(arguments='{"ticker": "AAPL"}', name='get_stock_price'), type='function')])
🤖 Coordinator routed task to -> get_weather_forecast({'location': 'Tokyo, Japan'})
🤖 Coordinator routed task to -> get_stock_price({'ticker': 'AAPL'})

✨ Final Agent Response:
The current weather in Tokyo is 23.61°C with overcast clouds and a humidity level of 76%. 

The current price of AAPL stock is $341.07.
